In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile


# For unzip and save in Google Drive
# Define file paths
zip_file_path = '/content/drive/My Drive/mobile_gan.zip'  # Path to your zip file on Google Drive
unzip_dir = '/content/drive/My Drive'  # Directory to unzip the file to

# Check if the unzip directory exists, if not, create it
if not os.path.exists(unzip_dir):
    os.makedirs(unzip_dir)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

print(f"Unzipped file saved to: {unzip_dir}")

KeyboardInterrupt: 

## Check for 100 epoches

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.utils import save_image
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import json
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import warnings
warnings.filterwarnings('ignore')


# CONFIGURATION PATHS - MODIFY THESE ACCORDING TO YOUR SETUP

# Google Drive base path
DRIVE_BASE_PATH = "/content/drive/MyDrive/mobile_gan"

# Input paths
OCCLUDED_IMAGES_PATH = os.path.join(DRIVE_BASE_PATH, "peper_gan_occluded")
ORIGINAL_IMAGES_PATH = os.path.join(DRIVE_BASE_PATH, "peper_gan_og")

# Output paths - All outputs will be saved to Google Drive
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, "outputs", "leaf_gan_output")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
SAMPLE_DIR = os.path.join(OUTPUT_DIR, "samples")
METRICS_DIR = os.path.join(OUTPUT_DIR, "metrics")
WEIGHTS_DIR = os.path.join(OUTPUT_DIR, "weights")
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOGS_DIR = os.path.join(OUTPUT_DIR, "logs")

# Training parameters
IMG_SIZE = 256
BATCH_SIZE = 8  # Small batch size for limited data
NUM_EPOCHS = 100
LEARNING_RATE = 0.0002
BETA1 = 0.5
LAMBDA_L1 = 100  # L1 loss weight
LAMBDA_PERCEPTUAL = 10  # Perceptual loss weight

# Create all output directories
def create_directories():
    """Create all necessary directories in Google Drive"""
    directories = [
        DRIVE_BASE_PATH,
        os.path.join(DRIVE_BASE_PATH, "data"),
        OCCLUDED_IMAGES_PATH,
        ORIGINAL_IMAGES_PATH,
        OUTPUT_DIR,
        MODELS_DIR,
        SAMPLE_DIR,
        METRICS_DIR,
        WEIGHTS_DIR,
        CHECKPOINTS_DIR,
        LOGS_DIR
    ]

    for directory in directories:
        os.makedirs(directory, exist_ok=True)
        print(f"Created/Verified directory: {directory}")

create_directories()


# LIGHTWEIGHT GENERATOR (Optimized for Edge Devices)


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, stride, padding, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.relu(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(channels, channels)
        self.conv2 = DepthwiseSeparableConv(channels, channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        return out + residual

class TinyUNet(nn.Module):
    """Lightweight U-Net Generator optimized for edge devices"""
    def __init__(self, in_channels=3, out_channels=3, base_channels=16):
        super().__init__()

        # Encoder
        self.enc1 = self._make_encoder_block(in_channels, base_channels)
        self.enc2 = self._make_encoder_block(base_channels, base_channels * 2)
        self.enc3 = self._make_encoder_block(base_channels * 2, base_channels * 4)
        self.enc4 = self._make_encoder_block(base_channels * 4, base_channels * 8)

        # Bottleneck with residual blocks
        self.bottleneck = nn.Sequential(
            ResidualBlock(base_channels * 8),
            ResidualBlock(base_channels * 8),
        )

        # Decoder
        self.dec4 = self._make_decoder_block(base_channels * 8, base_channels * 4)
        self.dec3 = self._make_decoder_block(base_channels * 8, base_channels * 2)  # 8 = 4 + 4 (skip connection)
        self.dec2 = self._make_decoder_block(base_channels * 4, base_channels)      # 4 = 2 + 2
        self.dec1 = self._make_decoder_block(base_channels * 2, base_channels)      # 2 = 1 + 1

        # Final layer
        self.final = nn.Sequential(
            nn.Conv2d(base_channels, out_channels, 3, padding=1),
            nn.Tanh()
        )

    def _make_encoder_block(self, in_ch, out_ch):
        return nn.Sequential(
            DepthwiseSeparableConv(in_ch, out_ch),
            DepthwiseSeparableConv(out_ch, out_ch),
            nn.MaxPool2d(2)
        )

    def _make_decoder_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2),
            DepthwiseSeparableConv(out_ch, out_ch),
            DepthwiseSeparableConv(out_ch, out_ch)
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)  # 128x128
        e2 = self.enc2(e1)  # 64x64
        e3 = self.enc3(e2)  # 32x32
        e4 = self.enc4(e3)  # 16x16

        # Bottleneck
        b = self.bottleneck(e4)

        # Decoder with skip connections
        d4 = self.dec4(b)  # 32x32
        d4 = torch.cat([d4, e3], dim=1)

        d3 = self.dec3(d4)  # 64x64
        d3 = torch.cat([d3, e2], dim=1)

        d2 = self.dec2(d3)  # 128x128
        d2 = torch.cat([d2, e1], dim=1)

        d1 = self.dec1(d2)  # 256x256

        return self.final(d1)

# LIGHTWEIGHT DISCRIMINATOR (70x70 PatchGAN)

class PatchGAN(nn.Module):
    """70x70 PatchGAN Discriminator"""
    def __init__(self, in_channels=6, base_channels=32):  # 6 = 3 (input) + 3 (target)
        super().__init__()

        self.model = nn.Sequential(
            # Layer 1: 256x256 -> 128x128
            nn.Conv2d(in_channels, base_channels, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 2: 128x128 -> 64x64
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 3: 64x64 -> 32x32
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # Layer 4: 32x32 -> 31x31
            nn.Conv2d(base_channels * 4, base_channels * 8, 4, 1, 1, bias=False),
            nn.BatchNorm2d(base_channels * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # Final layer: 31x31 -> 30x30
            nn.Conv2d(base_channels * 8, 1, 4, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, input_img, target_img):
        x = torch.cat([input_img, target_img], dim=1)
        return self.model(x)


# DATASET CLASS

class LeafDataset(Dataset):
    def __init__(self, occluded_paths, original_paths, transform=None):
        self.occluded_paths = occluded_paths
        self.original_paths = original_paths
        self.transform = transform

    def __len__(self):
        return len(self.occluded_paths)

    def __getitem__(self, idx):
        # Load images
        occluded_img = Image.open(self.occluded_paths[idx]).convert('RGB')
        original_img = Image.open(self.original_paths[idx]).convert('RGB')

        if self.transform:
            occluded_img = self.transform(occluded_img)
            original_img = self.transform(original_img)

        return occluded_img, original_img


# PERCEPTUAL LOSS (VGG-based)

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Use a lightweight feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.ReLU(inplace=True),
        )

        # Initialize with small random weights
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, pred, target):
        pred_features = self.features(pred)
        target_features = self.features(target)
        return F.mse_loss(pred_features, target_features)


# COLOR CONSISTENCY LOSS
class ColorConsistencyLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, pred, target, mask=None):
        """
        Ensures color consistency between predicted and target images
        mask: optional mask to focus on specific regions
        """
        if mask is not None:
            pred_masked = pred * mask
            target_masked = target * mask
            return F.mse_loss(pred_masked, target_masked)
        else:
            # Global color consistency
            pred_mean = torch.mean(pred, dim=[2, 3], keepdim=True)
            target_mean = torch.mean(target, dim=[2, 3], keepdim=True)
            return F.mse_loss(pred_mean, target_mean)


# LOGGING CLASS
class TrainingLogger:
    def __init__(self, log_file_path):
        self.log_file_path = log_file_path
        self.log_data = []

    def log(self, message):
        """Log message to both console and file"""
        print(message)
        self.log_data.append(message)

        # Append to log file
        with open(self.log_file_path, 'a') as f:
            f.write(f"{message}\n")

    def save_log(self):
        """Save complete log to file"""
        with open(self.log_file_path, 'w') as f:
            for message in self.log_data:
                f.write(f"{message}\n")


# UTILITY FUNCTIONS
def load_image_pairs():
    """Load and pair occluded and original images"""
    if not os.path.exists(OCCLUDED_IMAGES_PATH) or not os.path.exists(ORIGINAL_IMAGES_PATH):
        print(f"Error: Image directories not found!")
        print(f"Occluded: {OCCLUDED_IMAGES_PATH}")
        print(f"Original: {ORIGINAL_IMAGES_PATH}")
        print("\nPlease ensure your Google Drive structure is:")
        print(f"{DRIVE_BASE_PATH}/")
        print("├── data/")
        print("│   ├── occluded_images/")
        print("│   └── original_images/")
        return [], []

    occluded_files = sorted([f for f in os.listdir(OCCLUDED_IMAGES_PATH)
                            if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    original_files = sorted([f for f in os.listdir(ORIGINAL_IMAGES_PATH)
                            if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    # Ensure we have pairs
    if len(occluded_files) != len(original_files):
        print(f"Warning: Mismatch in number of images - Occluded: {len(occluded_files)}, Original: {len(original_files)}")
        min_length = min(len(occluded_files), len(original_files))
        occluded_files = occluded_files[:min_length]
        original_files = original_files[:min_length]

    occluded_paths = [os.path.join(OCCLUDED_IMAGES_PATH, f) for f in occluded_files]
    original_paths = [os.path.join(ORIGINAL_IMAGES_PATH, f) for f in original_files]

    return occluded_paths, original_paths

def calculate_metrics(pred, target):
    """Calculate PSNR and SSIM metrics"""
    pred_np = pred.cpu().numpy().transpose(1, 2, 0)
    target_np = target.cpu().numpy().transpose(1, 2, 0)

    # Convert to [0, 1] range
    pred_np = (pred_np + 1) / 2
    target_np = (target_np + 1) / 2

    # Clip values to ensure they're in valid range
    pred_np = np.clip(pred_np, 0, 1)
    target_np = np.clip(target_np, 0, 1)

    try:
        # Calculate PSNR
        psnr_val = psnr(target_np, pred_np, data_range=1.0)

        # Calculate SSIM
        ssim_val = ssim(target_np, pred_np, data_range=1.0, channel_axis=2)

        return psnr_val, ssim_val
    except Exception as e:
        print(f"Error calculating metrics: {e}")
        return 0.0, 0.0

def save_sample_images(generator, test_loader, epoch, device, logger):
    """Save sample images during training"""
    generator.eval()
    try:
        with torch.no_grad():
            occluded, original = next(iter(test_loader))
            occluded = occluded.to(device)
            original = original.to(device)

            fake = generator(occluded)

            # Save comparison - Input | Generated | Target
            comparison = torch.cat([occluded, fake, original], dim=3)
            save_path = os.path.join(SAMPLE_DIR, f'epoch_{epoch:03d}.png')
            save_image(comparison, save_path, nrow=1, normalize=True)

            logger.log(f"Sample images saved: {save_path}")
    except Exception as e:
        logger.log(f"Error saving sample images: {e}")

def count_parameters(model):
    """Count model parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def save_model_checkpoint(generator, discriminator, optimizer_g, optimizer_d, epoch, history, logger):
    """Save comprehensive model checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'optimizer_g_state_dict': optimizer_g.state_dict(),
        'optimizer_d_state_dict': optimizer_d.state_dict(),
        'history': history,
        'model_config': {
            'img_size': IMG_SIZE,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'lambda_l1': LAMBDA_L1,
            'lambda_perceptual': LAMBDA_PERCEPTUAL
        }
    }

    checkpoint_path = os.path.join(CHECKPOINTS_DIR, f'checkpoint_epoch_{epoch:03d}.pth')
    torch.save(checkpoint, checkpoint_path)
    logger.log(f"Checkpoint saved: {checkpoint_path}")

    # Also save as latest checkpoint
    latest_path = os.path.join(CHECKPOINTS_DIR, 'latest_checkpoint.pth')
    torch.save(checkpoint, latest_path)

# TRAINING FUNCTION

def train_gan():
    # Initialize logger
    log_file_path = os.path.join(LOGS_DIR, 'training_log.txt')
    logger = TrainingLogger(log_file_path)

    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logger.log(f"Using device: {device}")

    # Load data
    logger.log("Loading image pairs...")
    occluded_paths, original_paths = load_image_pairs()

    if len(occluded_paths) == 0:
        logger.log("No image pairs found. Please check your data paths.")
        return

    logger.log(f"Found {len(occluded_paths)} image pairs")

    # Split data (80% train, 20% test for limited data)
    train_occ, test_occ, train_orig, test_orig = train_test_split(
        occluded_paths, original_paths, test_size=0.2, random_state=42
    )

    logger.log(f"Training samples: {len(train_occ)}")
    logger.log(f"Testing samples: {len(test_occ)}")

    # Data transforms
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # [-1, 1]
    ])

    # Datasets and dataloaders
    train_dataset = LeafDataset(train_occ, train_orig, transform)
    test_dataset = LeafDataset(test_occ, test_orig, transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Models
    generator = TinyUNet().to(device)
    discriminator = PatchGAN().to(device)

    logger.log(f"Generator parameters: {count_parameters(generator):,}")
    logger.log(f"Discriminator parameters: {count_parameters(discriminator):,}")
    logger.log(f"Total parameters: {count_parameters(generator) + count_parameters(discriminator):,}")

    # Loss functions
    criterion_gan = nn.BCELoss()
    criterion_l1 = nn.L1Loss()
    criterion_perceptual = PerceptualLoss().to(device)
    criterion_color = ColorConsistencyLoss()

    # Optimizers
    optimizer_g = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
    optimizer_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

    # Training history
    history = {
        'g_loss': [], 'd_loss': [], 'psnr': [], 'ssim': [],
        'g_gan_loss': [], 'g_l1_loss': [], 'g_perceptual_loss': [], 'g_color_loss': []
    }

    logger.log("Starting training...")
    logger.log(f"Training configuration:")
    logger.log(f"  Epochs: {NUM_EPOCHS}")
    logger.log(f"  Batch size: {BATCH_SIZE}")
    logger.log(f"  Learning rate: {LEARNING_RATE}")
    logger.log(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")
    logger.log(f"  Lambda L1: {LAMBDA_L1}")
    logger.log(f"  Lambda Perceptual: {LAMBDA_PERCEPTUAL}")

    for epoch in range(NUM_EPOCHS):
        generator.train()
        discriminator.train()

        epoch_g_loss = 0
        epoch_d_loss = 0
        epoch_g_gan_loss = 0
        epoch_g_l1_loss = 0
        epoch_g_perceptual_loss = 0
        epoch_g_color_loss = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')

        for i, (occluded, original) in enumerate(pbar):
            occluded = occluded.to(device)
            original = original.to(device)

            batch_size = occluded.size(0)

            # Real and fake labels
            real_label = torch.ones(batch_size, 1, 30, 30).to(device)  # PatchGAN output size
            fake_label = torch.zeros(batch_size, 1, 30, 30).to(device)

            
            # Train Discriminator
            optimizer_d.zero_grad()

            # Real images
            real_pred = discriminator(occluded, original)
            d_real_loss = criterion_gan(real_pred, real_label)

            # Fake images
            fake_images = generator(occluded)
            fake_pred = discriminator(occluded, fake_images.detach())
            d_fake_loss = criterion_gan(fake_pred, fake_label)

            d_loss = (d_real_loss + d_fake_loss) * 0.5
            d_loss.backward()
            optimizer_d.step()

            
            # Train Generator
            optimizer_g.zero_grad()

            # GAN loss
            fake_pred = discriminator(occluded, fake_images)
            g_gan_loss = criterion_gan(fake_pred, real_label)

            # L1 loss
            g_l1_loss = criterion_l1(fake_images, original)

            # Perceptual loss
            g_perceptual_loss = criterion_perceptual(fake_images, original)

            # Color consistency loss
            g_color_loss = criterion_color(fake_images, original)

            # Total generator loss
            g_loss = g_gan_loss + LAMBDA_L1 * g_l1_loss + LAMBDA_PERCEPTUAL * g_perceptual_loss + g_color_loss
            g_loss.backward()
            optimizer_g.step()

            # Accumulate losses
            epoch_g_loss += g_loss.item()
            epoch_d_loss += d_loss.item()
            epoch_g_gan_loss += g_gan_loss.item()
            epoch_g_l1_loss += g_l1_loss.item()
            epoch_g_perceptual_loss += g_perceptual_loss.item()
            epoch_g_color_loss += g_color_loss.item()

            pbar.set_postfix({
                'G_Loss': f'{g_loss.item():.4f}',
                'D_Loss': f'{d_loss.item():.4f}'
            })

        # Calculate average losses
        avg_g_loss = epoch_g_loss / len(train_loader)
        avg_d_loss = epoch_d_loss / len(train_loader)
        avg_g_gan_loss = epoch_g_gan_loss / len(train_loader)
        avg_g_l1_loss = epoch_g_l1_loss / len(train_loader)
        avg_g_perceptual_loss = epoch_g_perceptual_loss / len(train_loader)
        avg_g_color_loss = epoch_g_color_loss / len(train_loader)

        # Evaluate on test set
        generator.eval()
        test_psnr = 0
        test_ssim = 0

        with torch.no_grad():
            for occluded, original in test_loader:
                occluded = occluded.to(device)
                original = original.to(device)

                fake = generator(occluded)

                # Calculate metrics for each image in batch
                for j in range(occluded.size(0)):
                    psnr_val, ssim_val = calculate_metrics(fake[j], original[j])
                    test_psnr += psnr_val
                    test_ssim += ssim_val

        test_psnr /= len(test_dataset)
        test_ssim /= len(test_dataset)

        # Save history
        history['g_loss'].append(avg_g_loss)
        history['d_loss'].append(avg_d_loss)
        history['g_gan_loss'].append(avg_g_gan_loss)
        history['g_l1_loss'].append(avg_g_l1_loss)
        history['g_perceptual_loss'].append(avg_g_perceptual_loss)
        history['g_color_loss'].append(avg_g_color_loss)
        history['psnr'].append(test_psnr)
        history['ssim'].append(test_ssim)

        # Log epoch results
        logger.log(f'Epoch {epoch+1}/{NUM_EPOCHS}:')
        logger.log(f'  Total G_Loss: {avg_g_loss:.4f}, D_Loss: {avg_d_loss:.4f}')
        logger.log(f'  G_GAN: {avg_g_gan_loss:.4f}, G_L1: {avg_g_l1_loss:.4f}')
        logger.log(f'  G_Perceptual: {avg_g_perceptual_loss:.4f}, G_Color: {avg_g_color_loss:.4f}')
        logger.log(f'  PSNR: {test_psnr:.2f}, SSIM: {test_ssim:.4f}')

        # Save sample images every 10 epochs
        if (epoch + 1) % 10 == 0:
            save_sample_images(generator, test_loader, epoch + 1, device, logger)

        # Save checkpoint every 25 epochs
        if (epoch + 1) % 25 == 0:
            save_model_checkpoint(generator, discriminator, optimizer_g, optimizer_d,
                                epoch + 1, history, logger)
        # Save training history every epoch
        history_serializable = {}
        for key, values in history.items():
            history_serializable[key] = [float(x.item()) if hasattr(x, 'item') else float(x) for x in values]

        with open(os.path.join(METRICS_DIR, 'training_history.json'), 'w') as f:
            json.dump(history_serializable, f, indent=2)


    # Save final models
    logger.log("Saving final models...")
    torch.save(generator.state_dict(), os.path.join(MODELS_DIR, 'generator_final.pth'))
    torch.save(discriminator.state_dict(), os.path.join(MODELS_DIR, 'discriminator_final.pth'))

    # Save final checkpoint
    save_model_checkpoint(generator, discriminator, optimizer_g, optimizer_d,
                         NUM_EPOCHS, history, logger)
    # Save training history
    history_serializable = {}
    for key, values in history.items():
        history_serializable[key] = [float(x.item()) if hasattr(x, 'item') else float(x) for x in values]

    with open(os.path.join(METRICS_DIR, 'training_history_final.json'), 'w') as f:
         json.dump(history_serializable, f, indent=2)


    # Plot training curves
    plot_training_curves(history, logger)

    # Final evaluation
    final_evaluation(generator, test_loader, device, logger)

    logger.log("Training completed!")
    logger.save_log()

# EVALUATION FUNCTIONS

def plot_training_curves(history, logger):
    """Plot comprehensive training curves"""
    try:
        epochs = range(1, len(history['g_loss']) + 1)

        # Create a comprehensive plot with multiple subplots
        fig = plt.figure(figsize=(20, 15))

        # 1. Generator and Discriminator Loss
        ax1 = plt.subplot(3, 3, 1)
        ax1.plot(epochs, history['g_loss'], 'b-', label='Generator Loss', linewidth=2)
        ax1.plot(epochs, history['d_loss'], 'r-', label='Discriminator Loss', linewidth=2)
        ax1.set_title('Training Losses', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2. Generator Loss Components
        ax2 = plt.subplot(3, 3, 2)
        ax2.plot(epochs, history['g_gan_loss'], 'g-', label='GAN Loss', linewidth=2)
        ax2.plot(epochs, history['g_l1_loss'], 'm-', label='L1 Loss', linewidth=2)
        ax2.plot(epochs, history['g_perceptual_loss'], 'c-', label='Perceptual Loss', linewidth=2)
        ax2.plot(epochs, history['g_color_loss'], 'y-', label='Color Loss', linewidth=2)
        ax2.set_title('Generator Loss Components', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3. PSNR
        ax3 = plt.subplot(3, 3, 3)
        ax3.plot(epochs, history['psnr'], 'g-', linewidth=2)
        ax3.set_title('PSNR', fontsize=12, fontweight='bold')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('PSNR (dB)')
        ax3.grid(True, alpha=0.3)

        # 4. SSIM
        ax4 = plt.subplot(3, 3, 4)
        ax4.plot(epochs, history['ssim'], 'm-', linewidth=2)
        ax4.set_title('SSIM', fontsize=12, fontweight='bold')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('SSIM')
        ax4.grid(True, alpha=0.3)

        # 5. Combined Metrics
        ax5 = plt.subplot(3, 3, 5)
        ax5_twin = ax5.twinx()
        line1 = ax5.plot(epochs, history['psnr'], 'g-', label='PSNR', linewidth=2)
        line2 = ax5_twin.plot(epochs, history['ssim'], 'm-', label='SSIM', linewidth=2)
        ax5.set_title('Combined Metrics', fontsize=12, fontweight='bold')
        ax5.set_xlabel('Epoch')
        ax5.set_ylabel('PSNR (dB)', color='g')
        ax5_twin.set_ylabel('SSIM', color='m')

        # Combine legends
        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax5.legend(lines, labels, loc='upper left')
        ax5.grid(True, alpha=0.3)

        # 6. Loss Ratio (G/D)
        ax6 = plt.subplot(3, 3, 6)
        loss_ratio = [g/d if d != 0 else 0 for g, d in zip(history['g_loss'], history['d_loss'])]
        ax6.plot(epochs, loss_ratio, 'orange', linewidth=2)
        ax6.set_title('Generator/Discriminator Loss Ratio', fontsize=12, fontweight='bold')
        ax6.set_xlabel('Epoch')
        ax6.set_ylabel('G_Loss / D_Loss')
        ax6.grid(True, alpha=0.3)

        # 7. Moving Average of Losses (window=10)
        if len(epochs) > 10:
            ax7 = plt.subplot(3, 3, 7)
            window = 10
            g_loss_ma = [np.mean(history['g_loss'][max(0, i-window):i+1]) for i in range(len(history['g_loss']))]
            d_loss_ma = [np.mean(history['d_loss'][max(0, i-window):i+1]) for i in range(len(history['d_loss']))]
            ax7.plot(epochs, g_loss_ma, 'b-', label='Generator (MA)', linewidth=2)
            ax7.plot(epochs, d_loss_ma, 'r-', label='Discriminator (MA)', linewidth=2)
            ax7.set_title(f'Moving Average Losses (window={window})', fontsize=12, fontweight='bold')
            ax7.set_xlabel('Epoch')
            ax7.set_ylabel('Loss')
            ax7.legend()
            ax7.grid(True, alpha=0.3)

        # 8. PSNR and SSIM Improvement Rate
        ax8 = plt.subplot(3, 3, 8)
        if len(epochs) > 1:
            psnr_diff = [history['psnr'][i] - history['psnr'][i-1] for i in range(1, len(history['psnr']))]
            ssim_diff = [history['ssim'][i] - history['ssim'][i-1] for i in range(1, len(history['ssim']))]
            ax8.plot(epochs[1:], psnr_diff, 'g-', label='PSNR Δ', linewidth=2)
            ax8_twin = ax8.twinx()
            ax8_twin.plot(epochs[1:], ssim_diff, 'm-', label='SSIM Δ', linewidth=2)
            ax8.set_title('Metrics Improvement Rate', fontsize=12, fontweight='bold')
            ax8.set_xlabel('Epoch')
            ax8.set_ylabel('PSNR Change', color='g')
            ax8_twin.set_ylabel('SSIM Change', color='m')
            ax8.grid(True, alpha=0.3)

        # 9. Training Summary Stats
        ax9 = plt.subplot(3, 3, 9)
        ax9.axis('off')

        # Calculate summary statistics
        final_psnr = history['psnr'][-1] if history['psnr'] else 0
        final_ssim = history['ssim'][-1] if history['ssim'] else 0
        max_psnr = max(history['psnr']) if history['psnr'] else 0
        max_ssim = max(history['ssim']) if history['ssim'] else 0
        final_g_loss = history['g_loss'][-1] if history['g_loss'] else 0
        final_d_loss = history['d_loss'][-1] if history['d_loss'] else 0

        summary_text = f"""
Training Summary:
━━━━━━━━━━━━━━━━━━━━━━━━━
Final PSNR: {final_psnr:.2f} dB
Max PSNR: {max_psnr:.2f} dB
Final SSIM: {final_ssim:.4f}
Max SSIM: {max_ssim:.4f}
━━━━━━━━━━━━━━━━━━━━━━━━━
Final G Loss: {final_g_loss:.4f}
Final D Loss: {final_d_loss:.4f}
━━━━━━━━━━━━━━━━━━━━━━━━━
Total Epochs: {len(epochs)}
        """

        ax9.text(0.1, 0.9, summary_text, transform=ax9.transAxes, fontsize=11,
                verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

        plt.tight_layout(pad=3.0)

        # Save plots
        plots_path = os.path.join(METRICS_DIR, 'comprehensive_training_curves.png')
        plt.savefig(plots_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()

        # Create a simple version for quick viewing
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))

        ax1.plot(epochs, history['g_loss'], 'b-', label='Generator Loss')
        ax1.plot(epochs, history['d_loss'], 'r-', label='Discriminator Loss')
        ax1.set_title('Training Losses')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)

        ax2.plot(epochs, history['psnr'], 'g-')
        ax2.set_title('PSNR')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('PSNR (dB)')
        ax2.grid(True)

        ax3.plot(epochs, history['ssim'], 'm-')
        ax3.set_title('SSIM')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('SSIM')
        ax3.grid(True)

        ax4_twin = ax4.twinx()
        ax4.plot(epochs, history['psnr'], 'g-', label='PSNR')
        ax4_twin.plot(epochs, history['ssim'], 'm-', label='SSIM')
        ax4.set_title('Combined Metrics')
        ax4.set_xlabel('Epoch')
        ax4.set_ylabel('PSNR (dB)', color='g')
        ax4_twin.set_ylabel('SSIM', color='m')
        ax4.legend(loc='upper left')
        ax4_twin.legend(loc='upper right')
        ax4.grid(True)

        plt.tight_layout()
        simple_plots_path = os.path.join(METRICS_DIR, 'simple_training_curves.png')
        plt.savefig(simple_plots_path, dpi=300, bbox_inches='tight')
        plt.close()

        logger.log(f"Training curves saved:")
        logger.log(f"  Comprehensive: {plots_path}")
        logger.log(f"  Simple: {simple_plots_path}")

    except Exception as e:
        logger.log(f"Error plotting training curves: {e}")

def final_evaluation(generator, test_loader, device, logger):
    """Comprehensive evaluation on test set"""
    generator.eval()

    all_psnr = []
    all_ssim = []

    logger.log("Performing final evaluation...")

    with torch.no_grad():
        for i, (occluded, original) in enumerate(tqdm(test_loader, desc="Final Testing")):
            occluded = occluded.to(device)
            original = original.to(device)

            fake = generator(occluded)

            # Save detailed results for first 10 test samples
            if i < 10:
                for j in range(min(occluded.size(0), 5)):  # Max 5 per batch
                    sample_idx = i * occluded.size(0) + j

                    # Create detailed comparison
                    input_img = occluded[j:j+1]
                    generated_img = fake[j:j+1]
                    target_img = original[j:j+1]

                    # Calculate metrics for this sample
                    psnr_val, ssim_val = calculate_metrics(fake[j], original[j])

                    # Create comparison grid
                    comparison = torch.cat([input_img, generated_img, target_img], dim=3)

                    # Add text labels (this would need PIL for text, simplified here)
                    save_path = os.path.join(SAMPLE_DIR, f'final_detailed_sample_{sample_idx:02d}_PSNR_{psnr_val:.1f}_SSIM_{ssim_val:.3f}.png')
                    save_image(comparison, save_path, nrow=1, normalize=True)

            # Calculate metrics for all samples
            for j in range(occluded.size(0)):
                psnr_val, ssim_val = calculate_metrics(fake[j], original[j])
                all_psnr.append(psnr_val)
                all_ssim.append(ssim_val)

    # Calculate comprehensive statistics
    mean_psnr = np.mean(all_psnr)
    std_psnr = np.std(all_psnr)
    median_psnr = np.median(all_psnr)
    min_psnr = np.min(all_psnr)
    max_psnr = np.max(all_psnr)

    mean_ssim = np.mean(all_ssim)
    std_ssim = np.std(all_ssim)
    median_ssim = np.median(all_ssim)
    min_ssim = np.min(all_ssim)
    max_ssim = np.max(all_ssim)

    # Save comprehensive evaluation results
    eval_results = {
        'psnr_statistics': {
            'mean': float(mean_psnr),
            'std': float(std_psnr),
            'median': float(median_psnr),
            'min': float(min_psnr),
            'max': float(max_psnr)
        },
        'ssim_statistics': {
            'mean': float(mean_ssim),
            'std': float(std_ssim),
            'median': float(median_ssim),
            'min': float(min_ssim),
            'max': float(max_ssim)
        },
        'all_psnr_values': [float(x.item()) if hasattr(x, 'item') else float(x) for x in all_psnr],
        'all_ssim_values': [float(x.item()) if hasattr(x, 'item') else float(x) for x in all_ssim],
        'test_samples_count': len(all_psnr)
    }

    eval_path = os.path.join(METRICS_DIR, 'comprehensive_evaluation.json')
    with open(eval_path, 'w') as f:
        json.dump(eval_results, f, indent=2)

    # Log results
    logger.log(f"\n{'='*60}")
    logger.log(f"COMPREHENSIVE FINAL EVALUATION RESULTS")
    logger.log(f"{'='*60}")
    logger.log(f"Test Samples: {len(all_psnr)}")
    logger.log(f"")
    logger.log(f"PSNR Statistics:")
    logger.log(f"  Mean: {mean_psnr:.2f} ± {std_psnr:.2f} dB")
    logger.log(f"  Median: {median_psnr:.2f} dB")
    logger.log(f"  Range: {min_psnr:.2f} - {max_psnr:.2f} dB")
    logger.log(f"")
    logger.log(f"SSIM Statistics:")
    logger.log(f"  Mean: {mean_ssim:.4f} ± {std_ssim:.4f}")
    logger.log(f"  Median: {median_ssim:.4f}")
    logger.log(f"  Range: {min_ssim:.4f} - {max_ssim:.4f}")
    logger.log(f"{'='*60}")

    # Create comprehensive visualization
    create_evaluation_plots(all_psnr, all_ssim, eval_results, logger)

    return eval_results

def create_evaluation_plots(all_psnr, all_ssim, eval_results, logger):
    """Create comprehensive evaluation plots"""
    try:
        fig = plt.figure(figsize=(16, 12))

        # 1. PSNR Distribution
        ax1 = plt.subplot(2, 3, 1)
        ax1.hist(all_psnr, bins=30, alpha=0.7, color='green', edgecolor='black')
        ax1.axvline(eval_results['psnr_statistics']['mean'], color='red', linestyle='--',
                   label=f"Mean: {eval_results['psnr_statistics']['mean']:.2f}")
        ax1.axvline(eval_results['psnr_statistics']['median'], color='blue', linestyle='--',
                   label=f"Median: {eval_results['psnr_statistics']['median']:.2f}")
        ax1.set_title('PSNR Distribution', fontsize=12, fontweight='bold')
        ax1.set_xlabel('PSNR (dB)')
        ax1.set_ylabel('Frequency')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2. SSIM Distribution
        ax2 = plt.subplot(2, 3, 2)
        ax2.hist(all_ssim, bins=30, alpha=0.7, color='magenta', edgecolor='black')
        ax2.axvline(eval_results['ssim_statistics']['mean'], color='red', linestyle='--',
                   label=f"Mean: {eval_results['ssim_statistics']['mean']:.4f}")
        ax2.axvline(eval_results['ssim_statistics']['median'], color='blue', linestyle='--',
                   label=f"Median: {eval_results['ssim_statistics']['median']:.4f}")
        ax2.set_title('SSIM Distribution', fontsize=12, fontweight='bold')
        ax2.set_xlabel('SSIM')
        ax2.set_ylabel('Frequency')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3. PSNR vs SSIM Scatter Plot
        ax3 = plt.subplot(2, 3, 3)
        scatter = ax3.scatter(all_psnr, all_ssim, alpha=0.6, c=range(len(all_psnr)), cmap='viridis')
        ax3.set_title('PSNR vs SSIM Correlation', fontsize=12, fontweight='bold')
        ax3.set_xlabel('PSNR (dB)')
        ax3.set_ylabel('SSIM')
        ax3.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax3, label='Sample Index')

        # Calculate correlation
        correlation = np.corrcoef(all_psnr, all_ssim)[0, 1]
        ax3.text(0.05, 0.95, f'Correlation: {correlation:.3f}', transform=ax3.transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        # 4. Box Plots
        ax4 = plt.subplot(2, 3, 4)
        box_data = [all_psnr, all_ssim]
        box_labels = ['PSNR (dB)', 'SSIM × 100']

        # Scale SSIM for better visualization
        scaled_ssim = [x * 100 for x in all_ssim]
        box_data[1] = scaled_ssim

        bp = ax4.boxplot(box_data, labels=box_labels, patch_artist=True)
        bp['boxes'][0].set_facecolor('green')
        bp['boxes'][1].set_facecolor('magenta')
        ax4.set_title('Metrics Box Plot', fontsize=12, fontweight='bold')
        ax4.set_ylabel('Value')
        ax4.grid(True, alpha=0.3)

        # 5. Cumulative Distribution
        ax5 = plt.subplot(2, 3, 5)
        sorted_psnr = np.sort(all_psnr)
        sorted_ssim = np.sort(all_ssim)
        y = np.arange(1, len(sorted_psnr) + 1) / len(sorted_psnr)

        ax5.plot(sorted_psnr, y, 'g-', linewidth=2, label='PSNR')
        ax5_twin = ax5.twinx()
        ax5_twin.plot(sorted_ssim, y, 'm-', linewidth=2, label='SSIM')

        ax5.set_title('Cumulative Distribution', fontsize=12, fontweight='bold')
        ax5.set_xlabel('PSNR (dB)')
        ax5.set_ylabel('Cumulative Probability', color='g')
        ax5_twin.set_ylabel('Cumulative Probability', color='m')
        ax5.grid(True, alpha=0.3)

        # 6. Performance Summary
        ax6 = plt.subplot(2, 3, 6)
        ax6.axis('off')

        # Quality assessment based on typical ranges
        psnr_quality = "Excellent" if eval_results['psnr_statistics']['mean'] > 30 else \
                      "Good" if eval_results['psnr_statistics']['mean'] > 25 else \
                      "Fair" if eval_results['psnr_statistics']['mean'] > 20 else "Poor"

        ssim_quality = "Excellent" if eval_results['ssim_statistics']['mean'] > 0.9 else \
                      "Good" if eval_results['ssim_statistics']['mean'] > 0.8 else \
                      "Fair" if eval_results['ssim_statistics']['mean'] > 0.7 else "Poor"

        summary_text = f"""
PERFORMANCE SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PSNR Analysis:
  Mean: {eval_results['psnr_statistics']['mean']:.2f} ± {eval_results['psnr_statistics']['std']:.2f} dB
  Quality: {psnr_quality}

SSIM Analysis:
  Mean: {eval_results['ssim_statistics']['mean']:.4f} ± {eval_results['ssim_statistics']['std']:.4f}
  Quality: {ssim_quality}

Dataset Info:
  Test Samples: {eval_results['test_samples_count']}
  Correlation: {correlation:.3f}

Recommendations:
  • PSNR > 25 dB = Good reconstruction
  • SSIM > 0.8 = Good structural similarity
  • Current model shows {'strong' if correlation > 0.7 else 'moderate' if correlation > 0.4 else 'weak'} correlation
        """

        ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes, fontsize=10,
                verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

        plt.tight_layout(pad=3.0)

        # Save evaluation plots
        eval_plots_path = os.path.join(METRICS_DIR, 'comprehensive_evaluation_plots.png')
        plt.savefig(eval_plots_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.close()

        logger.log(f"Evaluation plots saved: {eval_plots_path}")

    except Exception as e:
        logger.log(f"Error creating evaluation plots: {e}")


# MODEL SUMMARY AND ANALYSIS FUNCTIONS
def analyze_model_complexity():
    """Analyze and report model complexity"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    generator = TinyUNet().to(device)
    discriminator = PatchGAN().to(device)

    # Calculate model sizes
    gen_params = count_parameters(generator)
    disc_params = count_parameters(discriminator)
    total_params = gen_params + disc_params

    # Estimate model size in MB (assuming float32)
    model_size_mb = total_params * 4 / (1024 * 1024)

    # Analyze model layers
    gen_layers = sum(1 for _ in generator.modules())
    disc_layers = sum(1 for _ in discriminator.modules())

    complexity_analysis = {
        'generator': {
            'parameters': gen_params,
            'layers': gen_layers,
            'size_mb': gen_params * 4 / (1024 * 1024)
        },
        'discriminator': {
            'parameters': disc_params,
            'layers': disc_layers,
            'size_mb': disc_params * 4 / (1024 * 1024)
        },
        'total': {
            'parameters': total_params,
            'layers': gen_layers + disc_layers,
            'size_mb': model_size_mb
        }
    }

    # Save analysis
    analysis_path = os.path.join(METRICS_DIR, 'model_complexity_analysis.json')
    with open(analysis_path, 'w') as f:
        json.dump(complexity_analysis, f, indent=2)

    return complexity_analysis

def print_model_summary():
    """Print detailed model summary with complexity analysis"""
    complexity = analyze_model_complexity()

    print("="*80)
    print("LIGHTWEIGHT LEAF RECONSTRUCTION GAN - MODEL SUMMARY")
    print("="*80)

    print(f"\nGenerator (TinyUNet):")
    print(f"  Parameters: {complexity['generator']['parameters']:,}")
    print(f"  Layers: {complexity['generator']['layers']}")
    print(f"  Size: {complexity['generator']['size_mb']:.2f} MB")
    print(f"  Features:")
    print(f"    • Depthwise separable convolutions for efficiency")
    print(f"    • Residual blocks for better gradient flow")
    print(f"    • U-Net architecture with skip connections")
    print(f"    • Optimized for edge device deployment")

    print(f"\nDiscriminator (PatchGAN 70x70):")
    print(f"  Parameters: {complexity['discriminator']['parameters']:,}")
    print(f"  Layers: {complexity['discriminator']['layers']}")
    print(f"  Size: {complexity['discriminator']['size_mb']:.2f} MB")
    print(f"  Features:")
    print(f"    • Patch-based discrimination for detailed feedback")
    print(f"    • Lightweight architecture for faster training")

    print(f"\nTotal Model Complexity:")
    print(f"  Combined Parameters: {complexity['total']['parameters']:,}")
    print(f"  Combined Size: {complexity['total']['size_mb']:.2f} MB")
    print(f"  Memory Footprint: ~{complexity['total']['size_mb'] * 2:.1f} MB (including gradients)")

    print(f"\nTraining Configuration:")
    print(f"  Image Size: {IMG_SIZE}x{IMG_SIZE}")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  Epochs: {NUM_EPOCHS}")
    print(f"  Learning Rate: {LEARNING_RATE}")
    print(f"  Loss Weights: L1={LAMBDA_L1}, Perceptual={LAMBDA_PERCEPTUAL}")

    print(f"\nOutput Directories:")
    print(f"  Base Output: {OUTPUT_DIR}")
    print(f"  Models: {MODELS_DIR}")
    print(f"  Weights: {WEIGHTS_DIR}")
    print(f"  Samples: {SAMPLE_DIR}")
    print(f"  Metrics: {METRICS_DIR}")
    print(f"  Logs: {LOGS_DIR}")

    print("="*80)


# ============================================================================
# MODEL INFERENCE AND TESTING FUNCTIONS
# ============================================================================

def load_trained_model(model_path, device):
    """Load a trained generator model"""
    try:
        generator = TinyUNet().to(device)
        generator.load_state_dict(torch.load(model_path, map_location=device))
        generator.eval()
        print(f"Model loaded successfully from: {model_path}")
        return generator
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

def inference_on_image(generator, image_path, device, output_path=None):
    """Run inference on a single image"""
    try:
        # Load and preprocess image
        transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])

        image = Image.open(image_path).convert('RGB')
        input_tensor = transform(image).unsqueeze(0).to(device)

        # Generate reconstruction
        with torch.no_grad():
            output = generator(input_tensor)

        # Save result if output path provided
        if output_path:
            save_image(output, output_path, normalize=True)
            print(f"Result saved to: {output_path}")

        return output

    except Exception as e:
        print(f"Error during inference: {e}")
        return None

def batch_inference(generator, input_folder, output_folder, device):
    """Run inference on all images in a folder"""
    os.makedirs(output_folder, exist_ok=True)

    image_files = [f for f in os.listdir(input_folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    print(f"Processing {len(image_files)} images...")

    for img_file in tqdm(image_files, desc="Processing images"):
        input_path = os.path.join(input_folder, img_file)
        output_path = os.path.join(output_folder, f"reconstructed_{img_file}")

        inference_on_image(generator, input_path, device, output_path)

def create_comparison_grid(occluded_folder, generated_folder, original_folder, output_path, max_samples=20):
    """Create a comparison grid showing occluded -> generated -> original"""
    try:
        occluded_files = sorted([f for f in os.listdir(occluded_folder)
                               if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

        # Limit number of samples
        occluded_files = occluded_files[:max_samples]

        images = []
        for img_file in occluded_files:
            occluded_path = os.path.join(occluded_folder, img_file)
            generated_path = os.path.join(generated_folder, f"reconstructed_{img_file}")
            original_path = os.path.join(original_folder, img_file)

            if os.path.exists(generated_path) and os.path.exists(original_path):
                # Load images
                occluded = Image.open(occluded_path).resize((256, 256))
                generated = Image.open(generated_path).resize((256, 256))
                original = Image.open(original_path).resize((256, 256))

                # Convert to tensors
                transform = transforms.ToTensor()
                occ_tensor = transform(occluded)
                gen_tensor = transform(generated)
                orig_tensor = transform(original)

                # Combine horizontally
                combined = torch.cat([occ_tensor, gen_tensor, orig_tensor], dim=2)
                images.append(combined)

        if images:
            # Create grid
            grid = torch.stack(images[:min(len(images), max_samples)])
            save_image(grid, output_path, nrow=1, padding=2)
            print(f"Comparison grid saved: {output_path}")
        else:
            print("No matching image triplets found for comparison grid")

    except Exception as e:
        print(f"Error creating comparison grid: {e}")


# RESUME TRAINING FUNCTION
def resume_training(checkpoint_path):
    """Resume training from a checkpoint"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    try:
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Initialize models
        generator = TinyUNet().to(device)
        discriminator = PatchGAN().to(device)

        # Load model states
        generator.load_state_dict(checkpoint['generator_state_dict'])
        discriminator.load_state_dict(checkpoint['discriminator_state_dict'])

        # Initialize optimizers
        optimizer_g = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
        optimizer_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

        # Load optimizer states
        optimizer_g.load_state_dict(checkpoint['optimizer_g_state_dict'])
        optimizer_d.load_state_dict(checkpoint['optimizer_d_state_dict'])

        # Load training history
        history = checkpoint['history']
        start_epoch = checkpoint['epoch']

        print(f"Resuming training from epoch {start_epoch}")
        print(f"Previous best PSNR: {max(history['psnr']):.2f}")
        print(f"Previous best SSIM: {max(history['ssim']):.4f}")

        # Continue training (this would need to be implemented)
        # For now, just return the loaded components
        return generator, discriminator, optimizer_g, optimizer_d, history, start_epoch

    except Exception as e:
        print(f"Error resuming training: {e}")
        return None, None, None, None, None, 0


# MAIN EXECUTION AND UTILITY FUNCTIONS

def main():
    """Main function with options"""
    print("Lightweight Leaf Reconstruction GAN")
    print("="*50)

    # Print model summary
    print_model_summary()

    # Create project setup guide
    create_project_structure_guide()

    # Check if data directories exist
    if not os.path.exists(OCCLUDED_IMAGES_PATH) or not os.path.exists(ORIGINAL_IMAGES_PATH):
        print(f"\nWARNING: Data directories not found!")
        print(f"Expected structure:")
        print(f"  Occluded images: {OCCLUDED_IMAGES_PATH}")
        print(f"  Original images: {ORIGINAL_IMAGES_PATH}")
        print(f"\nPlease check the PROJECT_SETUP_GUIDE.md for detailed instructions.")

        # Ask user if they want to continue anyway (for setup purposes)
        response = input("\nContinue with training anyway? (y/n): ").lower().strip()
        if response != 'y':
            print("Setup complete. Please add your data and run again.")
            return

    # Start training
    print(f"\nStarting training...")
    print(f"All outputs will be saved to: {OUTPUT_DIR}")
    train_gan()

# Additional utility functions for post-training analysis

def calculate_model_flops():
    """Calculate approximate FLOPs for the model (simplified estimation)"""
    # This is a simplified FLOP calculation
    # For more accurate calculation, you'd need a library like ptflops

    # Rough estimation based on conv operations
    generator = TinyUNet()

    # Count conv layers and estimate FLOPs
    conv_layers = []
    for module in generator.modules():
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
            conv_layers.append(module)

    total_flops = 0
    input_size = IMG_SIZE

    # Very rough estimation - actual calculation would be more complex
    for layer in conv_layers:
        if hasattr(layer, 'kernel_size'):
            if isinstance(layer.kernel_size, tuple):
                k = layer.kernel_size[0] * layer.kernel_size[1]
            else:
                k = layer.kernel_size * layer.kernel_size

            flops = k * layer.in_channels * layer.out_channels * (input_size ** 2)
            total_flops += flops

            # Update input size (simplified)
            if hasattr(layer, 'stride'):
                stride = layer.stride if isinstance(layer.stride, int) else layer.stride[0]
                input_size = input_size // stride

    return total_flops

def create_training_report(history_path):
    """Create a comprehensive training report"""
    try:
        with open(history_path, 'r') as f:
            history = json.load(f)

        # Calculate training statistics
        final_epoch = len(history['g_loss'])
        best_psnr = max(history['psnr'])
        best_ssim = max(history['ssim'])
        best_psnr_epoch = history['psnr'].index(best_psnr) + 1
        best_ssim_epoch = history['ssim'].index(best_ssim) + 1

        final_g_loss = history['g_loss'][-1]
        final_d_loss = history['d_loss'][-1]

        # Create report
        report = f"""
# Lightweight Leaf Reconstruction GAN - Training Report

## Training Summary
- **Total Epochs**: {final_epoch}
- **Final Generator Loss**: {final_g_loss:.4f}
- **Final Discriminator Loss**: {final_d_loss:.4f}

## Best Performance
- **Best PSNR**: {best_psnr:.2f} dB (Epoch {best_psnr_epoch})
- **Best SSIM**: {best_ssim:.4f} (Epoch {best_ssim_epoch})

## Model Configuration
- **Image Size**: {IMG_SIZE}x{IMG_SIZE}
- **Batch Size**: {BATCH_SIZE}
- **Learning Rate**: {LEARNING_RATE}
- **L1 Weight**: {LAMBDA_L1}
- **Perceptual Weight**: {LAMBDA_PERCEPTUAL}

## Architecture Details
- **Generator**: TinyUNet with depthwise separable convolutions
- **Discriminator**: 70x70 PatchGAN
- **Total Parameters**: {analyze_model_complexity()['total']['parameters']:,}
- **Model Size**: {analyze_model_complexity()['total']['size_mb']:.2f} MB

## Training Stability
- **Loss Convergence**: {'Stable' if abs(history['g_loss'][-1] - history['g_loss'][-10]) < 0.1 else 'Unstable'}
- **Metric Improvement**: {'Yes' if history['psnr'][-1] > history['psnr'][0] else 'No'}

## Files Generated
- Models saved in: `{MODELS_DIR}`
- Training curves: `{METRICS_DIR}/comprehensive_training_curves.png`
- Sample images: `{SAMPLE_DIR}/`
- Full logs: `{LOGS_DIR}/training_log.txt`

Generated on: {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
        """

        report_path = os.path.join(METRICS_DIR, 'TRAINING_REPORT.md')
        with open(report_path, 'w') as f:
            f.write(report)

        print(f"Training report created: {report_path}")
        return report_path

    except Exception as e:
        print(f"Error creating training report: {e}")
        return None

# FINAL EXECUTION
if __name__ == "__main__":
    main()

# Additional functions that can be called after training:

def post_training_analysis():
    """Run comprehensive post-training analysis"""
    print("Running post-training analysis...")

    # Check if training history exists
    history_path = os.path.join(METRICS_DIR, 'training_history.json')
    if os.path.exists(history_path):
        # Create training report
        create_training_report(history_path)

        # Load and analyze final evaluation
        eval_path = os.path.join(METRICS_DIR, 'comprehensive_evaluation.json')
        if os.path.exists(eval_path):
            with open(eval_path, 'r') as f:
                eval_results = json.load(f)

            print(f"\nFinal Model Performance:")
            print(f"PSNR: {eval_results['psnr_statistics']['mean']:.2f} ± {eval_results['psnr_statistics']['std']:.2f} dB")
            print(f"SSIM: {eval_results['ssim_statistics']['mean']:.4f} ± {eval_results['ssim_statistics']['std']:.4f}")
    else:
        print("No training history found. Please run training first.")

def quick_test(image_path):
    """Quick test on a single image"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Check if final model exists
    model_path = os.path.join(MODELS_DIR, 'generator_final.pth')
    if os.path.exists(model_path):
        generator = load_trained_model(model_path, device)
        if generator:
            output_path = os.path.join(SAMPLE_DIR, 'quick_test_result.png')
            result = inference_on_image(generator, image_path, device, output_path)
            if result is not None:
                print(f"Quick test completed. Result saved to: {output_path}")
            return result
    else:
        print(f"No trained model found at: {model_path}")
        print("Please run training first.")
        return None

Created/Verified directory: /content/drive/MyDrive/mobile_gan
Created/Verified directory: /content/drive/MyDrive/mobile_gan/data
Created/Verified directory: /content/drive/MyDrive/mobile_gan/peper_gan_occluded
Created/Verified directory: /content/drive/MyDrive/mobile_gan/peper_gan_og
Created/Verified directory: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output
Created/Verified directory: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/models
Created/Verified directory: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/samples
Created/Verified directory: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/metrics
Created/Verified directory: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/weights
Created/Verified directory: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/checkpoints
Created/Verified directory: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/logs
Lightweight Leaf Reconstruction GAN
LIGHTWEIGHT LEAF RECONSTRUCTI

Epoch 1/100: 100%|██████████| 250/250 [00:54<00:00,  4.61it/s, G_Loss=13.7388, D_Loss=0.0762]


Epoch 1/100:
  Total G_Loss: 25.9460, D_Loss: 0.1884
  G_GAN: 2.8488, G_L1: 0.2304
  G_Perceptual: 0.0013, G_Color: 0.0463
  PSNR: 22.06, SSIM: 0.4294


Epoch 2/100: 100%|██████████| 250/250 [00:38<00:00,  6.42it/s, G_Loss=13.0777, D_Loss=0.0451]


Epoch 2/100:
  Total G_Loss: 12.0122, D_Loss: 0.3039
  G_GAN: 2.6475, G_L1: 0.0936
  G_Perceptual: 0.0004, G_Color: 0.0034
  PSNR: 23.23, SSIM: 0.6846


...

Epoch 100/100: 100%|██████████| 250/250 [00:38<00:00,  6.46it/s, G_Loss=18.4831, D_Loss=0.0000]


Epoch 100/100:
  Total G_Loss: 18.5223, D_Loss: 0.0000
  G_GAN: 12.4018, G_L1: 0.0612
  G_Perceptual: 0.0004, G_Color: 0.0007
  PSNR: 22.28, SSIM: 0.8168
Sample images saved: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/samples/epoch_100.png
Checkpoint saved: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/checkpoints/checkpoint_epoch_100.pth
Saving final models...
Checkpoint saved: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/checkpoints/checkpoint_epoch_100.pth
Training curves saved:
  Comprehensive: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/metrics/comprehensive_training_curves.png
  Simple: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/metrics/simple_training_curves.png
Performing final evaluation...


Final Testing: 100%|██████████| 63/63 [00:14<00:00,  4.43it/s]



COMPREHENSIVE FINAL EVALUATION RESULTS
Test Samples: 500

PSNR Statistics:
  Mean: 22.28 ± 3.94 dB
  Median: 22.19 dB
  Range: 13.99 - 35.01 dB

SSIM Statistics:
  Mean: 0.8168 ± 0.0603
  Median: 0.8268
  Range: 0.6375 - 0.9377
Evaluation plots saved: /content/drive/MyDrive/mobile_gan/outputs/leaf_gan_output/metrics/comprehensive_evaluation_plots.png
Training completed!


"\n# After training, you can use these functions:\n\n# 1. Run post-training analysis\npost_training_analysis()\n\n# 2. Test on a single image\nquick_test('/path/to/test/image.jpg')\n\n# 3. Batch processing\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\ngenerator = load_trained_model(os.path.join(MODELS_DIR, 'generator_final.pth'), device)\nbatch_inference(generator, '/path/to/input/folder', '/path/to/output/folder', device)\n\n# 4. Create comparison grid\ncreate_comparison_grid('/path/to/occluded', '/path/to/generated', '/path/to/original', 'comparison.png')\n\n# 5. Resume training from checkpoint\nresume_components = resume_training(os.path.join(CHECKPOINTS_DIR, 'latest_checkpoint.pth'))\n"